<a href="https://colab.research.google.com/github/yamamoto-yuuichi/Streamlit-x-Knowledge-Graph/blob/main/%E5%88%B6%E5%BE%A1%E5%9B%9E%E8%B7%AF%E3%81%AE%E8%87%AA%E5%8B%95%E6%A4%9C%E8%A8%BC%E3%81%A8%E3%82%B0%E3%83%A9%E3%83%95%E5%8F%AF%E8%A6%96%E5%8C%96%E3%83%84%E3%83%BC%E3%83%AB_ipynb_%E3%81%AE%E3%82%B3%E3%83%94%E3%83%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install networkx
!pip install pyvis

# 日本語表示用のフォント（Noto Sans CJK）をインストールする
!apt-get install -y fonts-noto-cjk

In [ ]:
"""
制御回路シーケンス × グラフ理論：リレー回路の自動検査・可視化。
Excelのノードリスト・エッジリストから回路グラフを組み立て、次の3つを行う。
1. 直並列縮約による、各出力（コイル・ランプ）がONになる条件式の導出
2. Excelの検査リスト（安全ルール表）に基づく回路の検査
3. pyvisによる対話型グラフの表示（問題が見つかった部品は赤で強調）
処理全体の流れは、ファイル末尾のエントリポイントを参照。
実行前に、環境構築セル（!pip install / !apt-get のセル）を先に実行しておくこと。
"""

# --- 標準ライブラリ ---
import importlib
import re
import sys

# --- 外部ライブラリ ---
import matplotlib
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from IPython.display import HTML, display
from pyvis.network import Network

# ============================================================
# 設定値
#   入力ファイルの場所や見た目を変えるときは、原則ここだけを書き換えればよい。
# ============================================================

# 入力ファイル
_DRIVE_DIR = "/content/drive/MyDrive/作業フォルダ名"
FILE_PATH = f"{_DRIVE_DIR}/回路構成部品リスト_自己保持回路_正常.xlsx"
RULES_PATH = f"{_DRIVE_DIR}/検査リスト.xlsx"

# シート名
NODES_SHEET = "ノードリスト"
EDGES_SHEET = "エッジリスト"
RULES_SHEET = "ルール一覧"

# ルール一覧シートの見出し行の位置。0から数えるため、Excelの3行目なら 2 を指定する。
RULES_HEADER_ROW = 2

# 列名
NODE_ID_COL = "name"
SOURCE_COL = "始点"
TARGET_COL = "終点"
SEQ_COL = "ノード"

# ノード種別・エッジ種別
WIRE_TYPE = "wire"
LINK_TYPE = "link"
WIRE_NODE_KIND = "wire_node"
POWER_BUS_KIND = "power_bus"
COIL_KIND = "coil"
B_CONTACT_KIND = "b_contact"
NET_KINDS = ("wire_node", "power_bus")  # 線番と母線
OUTPUT_KINDS = ("coil", "lamp")  # 出力にあたる部品

# 解析
POWER_SOURCE = "P"
WIRE_NAME_PREFIX = "W"
NO_CONDITION = "(条件なし)"
DEFAULT_RULE_LEVEL = "注意"
EMPTY_EXPRS = ("", NO_CONDITION, None)

# 条件式から部品名だけを単語として取り出すための並び。
# 英数字とアンダースコアの連なりを1単語とみなす。
TOKEN_PATTERN = re.compile(r"[A-Za-z0-9_]+")

# 描画
LAYOUT_SEED = 0
LAYOUT_K = 0.8
SCALE = 500  # 座標の拡大率。ノードが重なる場合は大きくする
EDGE_WIDTH = 2  # 全エッジ共通の線幅
EDGE_COLOR = "green"  # エッジの線色。配線・連動とも同じ色にし、実線か点線かで区別する
NODE_COLOR_OK = "lightblue"  # 問題が見つからなかった部品の色
NODE_COLOR_NG = "#ffb3b3"  # 問題が見つかった部品の色（赤系で目立たせる）
NODE_SHAPE = "box"  # ラベルが内側に入る形状（circle / ellipse も可）
NODE_BORDER_WIDTH_NG = 3
NODE_FONT_SIZE = 14
EDGE_FONT_SIZE = 12
ARROW_SCALE = 0.4
NET_HEIGHT = "800px"
NET_WIDTH = "100%"
OUTPUT_HTML = "graph.html"

# インストールしたフォントを認識させるため、フォント一覧を再読み込みする
matplotlib.font_manager._load_fontmanager(try_read_cache=False)

# 既定フォントを日本語対応のNoto Sans CJK JPに設定する
plt.rcParams["font.family"] = "Noto Sans CJK JP"


# ============================================================
# 入力の読み込みと整形
# ============================================================

def load_tables(path: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    ノードリストとエッジリストのシートを読み込む。
    """
    return (
        pd.read_excel(path, sheet_name=NODES_SHEET),
        pd.read_excel(path, sheet_name=EDGES_SHEET),
    )


def clean_tables(
    df_nodes: pd.DataFrame, df_edges: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    空行を除き、ID列を文字列化して前後の空白を取り除く。
    空行の除去：Excelの空行はNaNとして読み込まれ、そのまま文字列化すると
    "nan" という名前のノードになるため先に取り除く。
    文字列化と前後の空白除去：セル末尾の空白が残ると "R_a1" と "R_a1 " が
    別ノードになるため strip する。
    """

    # ノードIDが空の行、始点または終点が空の行を捨てる。
    # dropna の結果は元の表の一部を指す場合があり、そのまま列へ代入すると
    # SettingWithCopyWarning が出るため、独立した表として明示的に複製する。
    df_nodes = df_nodes.dropna(subset=[NODE_ID_COL]).copy()
    df_edges = df_edges.dropna(subset=[SOURCE_COL, TARGET_COL]).copy()

    # ノードID列・始点列・終点列を整える
    df_nodes[NODE_ID_COL] = df_nodes[NODE_ID_COL].astype(str).str.strip()
    df_edges[SOURCE_COL] = df_edges[SOURCE_COL].astype(str).str.strip()
    df_edges[TARGET_COL] = df_edges[TARGET_COL].astype(str).str.strip()

    return df_nodes, df_edges


def report_id_consistency(df_nodes: pd.DataFrame, df_edges: pd.DataFrame) -> None:
    """
    両シートのIDが対応しているか確認し、結果を表示する。
    """
    defined = set(df_nodes[NODE_ID_COL])  # ノードリストに定義されたID

    # エッジリストで使われているID
    used = set(df_edges[SOURCE_COL]) | set(df_edges[TARGET_COL])
    unregistered = sorted(used - defined)  # ノードリストに無いID＝線番ノードのはず

    # 線番ノードとして自動登録される（想定内）
    print("エッジにあるがノードリストに未登録:", unregistered)

    # 線番の命名規則（Wで始まる）に合わないもの。
    # 部品名の打ち間違いはここに現れるため必ず確認する
    suspicious = [n for n in unregistered if not n.startswith(WIRE_NAME_PREFIX)]
    print("うち、線番らしくない名前:", suspicious if suspicious else "なし")

    # 孤立ノードになる（要対処）
    print("ノードリストにあるがエッジ未使用:", sorted(defined - used))


# ============================================================
# グラフの構築
# ============================================================

def build_graph(df_nodes: pd.DataFrame, df_edges: pd.DataFrame) -> nx.DiGraph:
    """
    ノードリストとエッジリストから有向グラフを組み立てる。
    注意：配線（type="wire"）の始点・終点はどちらの順で書いても結果は変わらない。
    実CADの接続リストには向きが記録されていないため、向きに依存しない処理にしてある。
    連動（type="link"）だけは「コイル → 接点」の向きに意味があるので、順序を守ること。
    """
    G = nx.DiGraph()  # 空の有向グラフを作る
    _add_nodes(G, df_nodes)
    _add_edges(G, df_edges)

    return G


def _add_nodes(G: nx.DiGraph, df_nodes: pd.DataFrame) -> None:
    """
    ノードリストの各行を、属性付きのノードとして追加する。
    """
    for _, row in df_nodes.iterrows():
        node_id = row[NODE_ID_COL]  # ノードの識別子を取り出す
        # ID列と連番列を除いた残りを属性にする
        attrs = row.drop([NODE_ID_COL, SEQ_COL], errors="ignore").dropna().to_dict()
        G.add_node(node_id, **attrs)


def _add_edges(G: nx.DiGraph, df_edges: pd.DataFrame) -> None:
    """
    エッジリストの各行を、属性付きのエッジとして追加する。
    ノードリストに載っていない端点は、線番ノードとみなして自動登録する。
    """
    for _, row in df_edges.iterrows():
        source = row[SOURCE_COL]
        target = row[TARGET_COL]
        for node_id in (source, target):
            if node_id not in G.nodes:  # ノードリストに載っていない＝線番ノード
                G.add_node(node_id, name=node_id, kind=WIRE_NODE_KIND)

        # 始点・終点以外の列を属性の辞書にする（空欄は除く）
        attrs = row.drop([SOURCE_COL, TARGET_COL], errors="ignore").dropna().to_dict()
        G.add_edge(source, target, **attrs)


def report_graph_summary(G: nx.DiGraph) -> None:
    """
    ノード数・エッジ数・kind属性の欠落・エッジ種別の内訳を表示する。
    """
    # 期待値と一致するか確認する
    print("ノード数:", G.number_of_nodes(), "エッジ数:", G.number_of_edges())
    missing_kind = [n for n, d in G.nodes(data=True) if "kind" not in d]
    print("kind属性が無いノード:", missing_kind if missing_kind else "なし")
    wire_edges = [
        (u, v) for u, v, d in G.edges(data=True) if d.get("type") == WIRE_TYPE
    ]
    link_edges = [
        (u, v) for u, v, d in G.edges(data=True) if d.get("type") == LINK_TYPE
    ]
    print(
        "配線エッジ:",
        len(wire_edges),
        "本 ／ 連動エッジ:",
        len(link_edges),
        "本",
        link_edges,
    )


def build_wire_graph(G: nx.DiGraph) -> nx.Graph:
    """
    配線エッジだけを取り出した無向グラフを作る。
    連動エッジは論理に無関係なので除く。
    """
    wire = nx.Graph()
    for u, v, d in G.edges(data=True):
        if d.get("type") == WIRE_TYPE:
            wire.add_edge(u, v)
    return wire


# ============================================================
# パート1：直並列縮約による条件式の導出
#   線番を「点」、部品を「線」と見立て、直列と並列を内側から順にまとめていく。
#   入れ子がどれだけ深くても、同じ2つの操作の繰り返しで解ける。
# ============================================================

def print_output_conditions(G: nx.DiGraph) -> None:
    """
    コイルとランプについて、ONになる条件式を求めて表示する。
    """
    print("\n各出力がONになる条件式（直並列縮約による導出）")
    for n, d in G.nodes(data=True):
        if d.get("kind") in OUTPUT_KINDS:  # 出力にあたる部品だけを対象にする
            print(f"  {n} : {condition_for(G, n)}")


def condition_for(G: nx.DiGraph, target: str, source: str = POWER_SOURCE) -> str | None:
    """
    target（コイルやランプ）がONになる条件式を求める。
    """
    wire = build_wire_graph(G)  # 配線グラフは1回だけ作り、以降の2関数で共用する
    element_graph, net_rep = build_element_graph(G, wire)

    # target自身を表す線（エッジ）を探す
    target_edges = [
        (u, v, k)
        for u, v, k, d in element_graph.edges(keys=True, data=True)
        if d["expr"] == target
    ]
    if not target_edges:
        return None
    u, v, k = target_edges[0]
    element_graph.remove_edge(u, v, k)  # target自身は条件に含めないので取り除く

    # 電源側を求める。配線の向きは使わない。
    up_net = find_upstream_net(G, wire, target, source)
    if up_net is None:
        print(
            f"警告: {target} の電源側を判定できませんでした。配線データを確認してください。"
        )
        upstream = u  # 判定できない場合は暫定的に片側を使う
    else:
        upstream = net_rep[up_net]
    src = net_rep[source]  # 電源の代表名
    keep = {src, upstream}  # この2点は残す
    element_graph = prune_dangling(element_graph, keep)  # 関係ない枝を刈る
    element_graph = reduce_series_parallel(element_graph, keep)  # 縮約する
    exprs = [
        d["expr"]
        for a, b, d in element_graph.edges(data=True)
        if {a, b} == {src, upstream}
    ]
    return exprs[0] if exprs else NO_CONDITION


def build_element_graph(
    G: nx.DiGraph, wire: nx.Graph
) -> tuple[nx.MultiGraph, dict[str, str]]:
    """
    線番を「点」、部品を「線」とするグラフに組み替える。
    """
    kind = nx.get_node_attributes(G, "kind")  # 各ノードの種類を取り出す
    netlike = {n for n in wire.nodes if kind.get(n) in NET_KINDS}
    net_rep = _merge_direct_nets(wire, netlike, kind)
    element_graph = nx.MultiGraph()  # 同じ2点間に複数の部品が並ぶため多重グラフを使う
    for n in G.nodes():
        if kind.get(n) in NET_KINDS:  # 部品だけを線として扱う
            continue
        nets = [net_rep[x] for x in wire.neighbors(n)]  # その部品がまたぐ2つの点
        if len(nets) != 2:
            continue
        element_graph.add_edge(nets[0], nets[1], expr=n)  # 部品名を式の断片として持たせる
    return element_graph, net_rep


def _merge_direct_nets(
    wire: nx.Graph, netlike: set[str], kind: dict[str, str]
) -> dict[str, str]:
    """
    部品を挟まず直結した線番どうし（P—W1 など）を、1つの代表名にまとめる。
    電気的に同じ点であるため統合する。母線があれば優先して代表にする。
    """
    merge = nx.Graph()
    merge.add_nodes_from(netlike)
    for u, v in wire.edges():
        if u in netlike and v in netlike:  # 両端とも線番＝部品を挟んでいない
            merge.add_edge(u, v)
    net_rep = {}  # 統合後の代表名を記録する辞書（元の名前 → 代表名）
    for group in nx.connected_components(merge):  # 統合されるグループごとに
        name = min(group, key=lambda x: (kind.get(x) != POWER_BUS_KIND, x))
        for member in group:
            net_rep[member] = name
    return net_rep


def find_upstream_net(
    G: nx.DiGraph, wire: nx.Graph, target: str, source: str = POWER_SOURCE
) -> str | None:
    """
    target（コイルやランプ）の電源側の線番を、配線の向きを使わずに求める。
    考え方：シーケンス図では負荷（コイル・ランプ）は必ずN母線側に置かれる。
    そのため出力部品をすべて取り除くと、接点だけの論理側がP母線につながって残る。
    targetの両側の線番のうち、この残った側にあるものが電源側である。
    回路は輪になっているため「電源からたどり着ける方」という単純な判定では
    両側とも該当してしまう。出力部品を外すことで、その輪を断ち切っている。
    """
    if target not in wire:
        return None
    nets = list(wire.neighbors(target))  # targetがまたぐ線番
    outputs = [n for n, d in G.nodes(data=True) if d.get("kind") in OUTPUT_KINDS]
    logic = wire.copy()
    logic.remove_nodes_from(outputs)  # 出力を取り除き、接点だけの論理側を残す

    # 電源側に残っている線番
    up = [n for n in nets if n in logic and nx.has_path(logic, source, n)]
    if len(up) == 1:
        return up[0]
    return None  # 判定できない場合（配線に誤りがある可能性）


def prune_dangling(element_graph: nx.MultiGraph, keep: set[str]) -> nx.MultiGraph:
    """
    行き止まりの枝を取り除く。keepに入れた点は残す。
    """
    changed = True
    while changed:  # 削るたびに新たな行き止まりが生まれるため繰り返す
        changed = False
        for n in list(element_graph.nodes()):
            # つながる先が1本以下＝行き止まり
            if n not in keep and element_graph.degree(n) <= 1:
                element_graph.remove_node(n)
                changed = True
    return element_graph


def reduce_series_parallel(
    element_graph: nx.MultiGraph, keep: set[str]
) -> nx.MultiGraph:
    """
    直列と並列の縮約を、変化しなくなるまで繰り返す。
    直列を優先し、縮約できたら先頭からやり直す。
    直列も並列もまとめられなくなったら終了する。
    """
    while _reduce_one_series(element_graph, keep) or _reduce_one_parallel(element_graph):
        pass
    return element_graph


def _reduce_one_series(element_graph: nx.MultiGraph, keep: set[str]) -> bool:
    """
    直列：部品が2つしかつながっていない点を1つ見つけ、その両側をANDでまとめる。
    """
    for n in list(element_graph.nodes()):
        if n in keep:  # 起点と終点は消してはいけない
            continue
        edges = list(element_graph.edges(n, keys=True, data=True))
        if len(edges) != 2:
            continue
        (u1, v1, _, d1), (u2, v2, _, d2) = edges
        a = v1 if u1 == n else u1  # 片側の相手
        b = v2 if u2 == n else u2  # もう片側の相手
        if a == b:  # 輪になる場合は直列ではないので飛ばす
            continue
        element_graph.remove_node(n)  # 間の点を消す
        # 両側を1つの部品（AND式）にまとめる
        element_graph.add_edge(a, b, expr=f"({d1['expr']} AND {d2['expr']})")
        return True
    return False


def _reduce_one_parallel(element_graph: nx.MultiGraph) -> bool:
    """
    並列：同じ2点をまたぐ部品が2つ以上ある箇所を1つ見つけ、ORでまとめる。
    """
    for u, v in {(min(a, b), max(a, b)) for a, b in element_graph.edges()}:
        keys = list(element_graph[u][v].keys())
        if len(keys) < 2:
            continue
        e1 = element_graph[u][v][keys[0]]["expr"]
        e2 = element_graph[u][v][keys[1]]["expr"]
        element_graph.remove_edge(u, v, keys[0])
        element_graph.remove_edge(u, v, keys[1])
        element_graph.add_edge(u, v, expr=f"({e1} OR {e2})")  # 1つの部品として扱う
        return True
    return False


# ============================================================
# パート2：安全ルールによる検査（これが価値になる出力）
#   「何を欠陥と見なすか」は 検査リスト.xlsx に表として分けてある。
#   本体はルールを実行する仕組みだけを持ち、判定内容には関与しない。
# ============================================================

def run_inspection(G: nx.DiGraph) -> tuple[list[str], set[str]]:
    """
    ルール表を読み込んで検査を実行し、判定結果を表示する。
    """
    print("\n安全ルールによる検査結果")
    rules = load_rules(RULES_PATH)  # Excelからルールを読み込む
    print(
        f"  適用するルール: {len(rules)}件 （{', '.join(r['title'] for r in rules)}）"
    )
    problems, flagged = check_circuit(G, rules)  # 検査を実行する
    print("  --- 判定 ---")
    if problems:
        for p in problems:
            print("   ", p)
    else:
        print("    問題は見つかりませんでした。")
    return problems, flagged


def load_rules(path: str) -> list[dict]:
    """
    Excelのルール表を読み込む。「有効」列に印があるものだけを返す。
    シートは、1行目に区分の説明、2行目は空行、3行目に列名、4行目以降にルールが
    並ぶ構成になっている。そのため header に RULES_HEADER_ROW を渡し、
    3行目を列名として読み込む。
    また、列名で必要な6列だけを参照するため、その右側にある説明用の列
    （ひとことで言うと、どう判定するか など）が増減しても動作に影響しない。
    """
    df = pd.read_excel(
        path, sheet_name=RULES_SHEET, header=RULES_HEADER_ROW
    ).fillna("")
    active = []
    for _, row in df.iterrows():
        if str(row["有効"]).strip() == "":  # 有効列が空欄なら飛ばす
            continue
        ctype = str(row["検査タイプ"]).strip()
        if ctype not in CHECK_TYPES:  # 表にない検査タイプは警告して飛ばす
            print(
                f"  警告: 検査タイプ「{ctype}」は未対応です"
                f"（ルール「{row['ルール名']}」を飛ばしました）"
            )
            continue
        active.append(
            {
                "title": str(row["ルール名"]).strip(),
                "level": str(row["深刻度"]).strip() or DEFAULT_RULE_LEVEL,
                "type": ctype,
                "param": str(row["パラメータ"]).strip(),
                "message": str(row["メッセージ"]).strip(),
            }
        )
    return active


def check_circuit(G: nx.DiGraph, rules: list[dict]) -> tuple[list[str], set[str]]:
    """
    全コイルについて、読み込んだ全ルールを順に適用する。
    """
    problems = []  # 見つかった問題をためるリスト
    flagged = set()  # 問題が見つかった部品の名前をためる集合（図の色分けに使う）
    b_contacts = [n for n, d in G.nodes(data=True) if d.get("kind") == B_CONTACT_KIND]
    for coil in [n for n, d in G.nodes(data=True) if d.get("kind") == COIL_KIND]:
        expr = condition_for(G, coil) or ""
        print(f"  {coil} の条件式: {expr}")
        ctx = RuleContext(G, coil, expr, b_contacts)
        for r in rules:
            if CHECK_TYPES[r["type"]](ctx, r["param"]):  # 問題ありなら True が返る
                # メッセージの {coil} などを実際の値に置き換える
                msg = r["message"].format(coil=coil, expr=expr, own=", ".join(ctx.own))
                problems.append(f"[{r['level']}] {msg}")
                flagged.add(coil)
    return problems, flagged


def tokenize_expr(expr: str) -> set[str]:
    """
    条件式を、部品名の単語の集合に分解する。
    条件式は "(BS2_b AND (BS1_a OR R_a1))" のように、AND/OR/カッコ/空白で
    区切られた部品名の並びである。含まれるかどうかの判定を文字列の部分一致
    （"R" in expr）で行うと、"R_a1" の中の "R" に、部品名 "R" が誤って
    ヒットする。判定は常に、この関数が返す単語の集合に対して行う。
    """
    return set(TOKEN_PATTERN.findall(expr))


class RuleContext:
    """
    ルールの判定に使う道具をまとめた入れ物。
    """

    def __init__(
        self, G: nx.DiGraph, coil: str, expr: str, b_contacts: list[str]
    ) -> None:
        self.G = G  # グラフ本体
        self.coil = coil  # 今調べているコイルの名前
        self.expr = expr  # そのコイルがONになる条件式（メッセージの表示用）
        self.expr_tokens = tokenize_expr(expr)  # 条件式を単語に分解したもの（判定用）
        self.b_contacts = b_contacts  # 回路内のb接点の名前一覧

        # そのコイルに連動する接点
        self.own = [
            v
            for u, v, d in G.edges(data=True)
            if u == coil and d.get("type") == LINK_TYPE
        ]
        self._wire = build_wire_graph(G)  # 判定用に配線だけの無向グラフを作っておく

    def names_of_kind(self, kind: str) -> list[str]:
        """
        指定した種類の部品の名前を集める。
        """
        return [n for n, d in self.G.nodes(data=True) if d.get("kind") == kind]

    def is_parallel(self, part: str) -> bool:
        """
        その部品が並列（OR）の位置にあるかを返す。
        """
        if part not in self._wire:
            return False
        part_nets = tuple(sorted(self._wire.neighbors(part)))
        if len(part_nets) != 2:
            return False
        for n, d in self.G.nodes(data=True):
            if n == part or d.get("kind") in NET_KINDS:
                continue
            if n in self._wire and tuple(sorted(self._wire.neighbors(n))) == part_nets:
                return True
        return False


# ---- 検査タイプごとの判定処理 ----
# 表の「検査タイプ」列の値と、末尾の CHECK_TYPES 辞書で対応づけている。
# 問題があれば True、問題なければ False を返す。
# 条件式に含まれるかどうかは、部分一致ではなく ctx.expr_tokens（単語の集合）で判定する。

def _type_contains_kind(ctx: RuleContext, param: str) -> bool:
    """
    条件式に、指定した種類の部品が含まれるか。
    """
    targets = ctx.names_of_kind(param)
    return not any(t in ctx.expr_tokens for t in targets)


def _type_not_contains_kind(ctx: RuleContext, param: str) -> bool:
    """
    条件式に、指定した種類の部品が含まれないか。
    """
    targets = ctx.names_of_kind(param)
    return any(t in ctx.expr_tokens for t in targets)


def _type_contains_name(ctx: RuleContext, param: str) -> bool:
    """
    条件式に、指定した名前が含まれるか。
    """
    return param not in ctx.expr_tokens


def _type_self_holding(ctx: RuleContext, param: str) -> bool:
    """
    コイル自身の接点が、起動回路の並列の位置にあるか。
    """
    if not ctx.own:
        return False  # 連動する接点が無いコイルは対象外
    return not any(c in ctx.expr_tokens and ctx.is_parallel(c) for c in ctx.own)


def _type_not_empty(ctx: RuleContext, param: str) -> bool:
    """
    条件式が求められているか。
    """
    return ctx.expr in EMPTY_EXPRS


# 「検査タイプ」列の値と判定関数の対応表。
# 新しい検査タイプを増やすときは、関数を追加してここに1行足す。
CHECK_TYPES = {
    "条件式に種類が含まれる": _type_contains_kind,
    "条件式に種類が含まれない": _type_not_contains_kind,
    "条件式に名前が含まれる": _type_contains_name,
    "自己保持している": _type_self_holding,
    "条件式が空でない": _type_not_empty,
}


# ============================================================
# 可視化
# ============================================================

def visualize(G: nx.DiGraph, flagged: set[str]) -> Network:
    """
    ラベルと座標を作り、対話型グラフを描画する。
    """
    node_labels = build_node_labels(G)
    edge_labels = build_edge_labels(G)
    # matplotlib版と同じ座標計算（seed=0, k=0.8 で毎回同じ配置）
    pos = nx.spring_layout(G, seed=LAYOUT_SEED, k=LAYOUT_K)
    net = build_network(G, node_labels, edge_labels, pos, flagged)
    render_html(net)
    return net


def build_node_labels(G: nx.DiGraph) -> dict[str, str]:
    """
    ノードに表示するラベル（ノード名＋属性）を作成する。
    注意：並列かどうか（AND/OR）はラベルに出さない。グラフの形が並列を示している
    ため情報が重複し、かつ「1つの部品」と「複数部品の直列のかたまり」が並列の
    場合には誤った表示になるため。
    """
    node_labels = {}
    for n, data in G.nodes(data=True):
        label_parts = [str(n)]  # まずノード名をラベルの先頭に入れる
        for key, value in data.items():
            if key == "name":  # nameはノード名と重複するので飛ばす
                continue
            label_parts.append(f"{key}: {value}")  # 「属性名: 値」の形にして追加する
        node_labels[n] = "\n".join(label_parts)  # 改行でつないで1つのラベルにする
    return node_labels


def build_edge_labels(G: nx.DiGraph) -> dict[tuple[str, str], str]:
    """
    エッジに表示するラベル（属性）を作成する。
    """
    edge_labels = {}
    for u, v, data in G.edges(data=True):
        label_parts = [f"{key}: {value}" for key, value in data.items()]
        if label_parts:  # 属性が1つ以上ある場合のみ
            edge_labels[(u, v)] = "\n".join(label_parts)
    return edge_labels


def build_network(
    G: nx.DiGraph, node_labels: dict, edge_labels: dict, pos: dict, flagged: set[str]
) -> Network:
    """
    pyvisの対話型グラフを組み立てる。
    """
    # 対話型グラフの描画領域を用意する
    net = Network(
        notebook=True,
        directed=True,
        cdn_resources="in_line",
        height=NET_HEIGHT,
        width=NET_WIDTH,
    )
    net.from_nx(G)  # networkxのグラフをpyvisに変換する
    net.options.edges.smooth.enabled = False  # エッジを直線にする
    _style_nodes(net, node_labels, pos, flagged)
    _style_edges(net, G, edge_labels)
    net.toggle_physics(False)  # 物理演算を停止し、上で与えた座標を保持する
    return net


def _style_nodes(net: Network, node_labels: dict, pos: dict, flagged: set[str]) -> None:
    """
    ノードのラベル・形状・色・座標を設定する。
    """
    for node in net.nodes:
        nid = node["id"]
        node["label"] = node_labels[nid]  # ノード名＋属性の複数行ラベル
        node["shape"] = NODE_SHAPE
        if nid in flagged:  # 検査で問題が見つかった部品
            node["color"] = NODE_COLOR_NG  # 赤系にしてひと目で分かるようにする
            node["borderWidth"] = NODE_BORDER_WIDTH_NG  # 枠線も太くして強調する
        else:
            node["color"] = NODE_COLOR_OK  # 問題なしは通常の色
        node["font"] = {"size": NODE_FONT_SIZE}
        x, y = pos[nid]  # matplotlib版と同じ位置に固定する
        node["x"] = float(x) * SCALE
        node["y"] = float(y) * SCALE


def _style_edges(net: Network, G: nx.DiGraph, edge_labels: dict) -> None:
    """
    エッジのラベル・線幅・線種・矢印を設定する。
    矢印は連動エッジ（コイル → 接点）にだけ付ける。
    配線エッジは向きに意味がないため矢印を出さない。矢印があると
    「電気がこの向きにしか流れない」という誤解を与えるため。
    """
    for edge in net.edges:
        key = (edge["from"], edge["to"])
        if key in edge_labels:
            edge["label"] = edge_labels[key]  # 属性の複数行ラベル

        # valueがあると幅が自動スケーリングされwidthが無視されるため削除する
        edge.pop("value", None)
        edge["width"] = EDGE_WIDTH  # 全エッジの線幅を統一する
        edge["chosen"] = False  # 選択・ホバー時に線幅が変化するのを止める
        edge_type = G.edges[key].get("type")  # そのエッジの種類（wire か link）
        edge["color"] = EDGE_COLOR  # 線色は種類によらず統一する

        # 連動エッジのみ点線にする。配線エッジは実線のまま
        edge["dashes"] = edge_type == LINK_TYPE
        edge["arrows"] = {
            "to": {"enabled": edge_type == LINK_TYPE, "scaleFactor": ARROW_SCALE}
        }
        edge["font"] = {"size": EDGE_FONT_SIZE, "align": "middle"}


def render_html(net: Network, path: str = OUTPUT_HTML) -> None:
    """
    HTMLファイルとして書き出し、Colab上に埋め込み表示する。
    """
    net.write_html(path)
    with open(path, "r", encoding="utf-8") as f:
        html = f.read()
    display(HTML(html))


# ============================================================
# エントリポイント
#   処理全体の流れ：読み込み → 整形 → グラフ構築 → 条件式導出 → 検査 → 可視化
# ============================================================

if __name__ == "__main__":
    df_nodes, df_edges = load_tables(FILE_PATH)
    df_nodes, df_edges = clean_tables(df_nodes, df_edges)
    report_id_consistency(df_nodes, df_edges)
    G = build_graph(df_nodes, df_edges)
    report_graph_summary(G)
    print_output_conditions(G)
    problems, flagged = run_inspection(G)
    net = visualize(G, flagged)